# Sorting 
    ada 3 jenis sorting yang kami gunakan :
    bubble sort , merge sort , dan shell sort 
berdasarakan medali yaitu sebagai berikut :
    (emas → perak → perunggu → harapan → partisipan)

# Cara penggunaan kode 
langsung jalankan kode dan jika ingin mengganti nama file ada pada  # file_path = "osn.csv" # 

# rumus untuk mencari single & Multi threading : 
1. Execution Time = End Time - Start Time
2. Speedup = Single-thread Time / Multi-thread Time
3. Performance Improvement (%) = ((Single-thread Time - Multi-thread Time) / Single-thread Time) × 100
4. Efficiency (%) = (Speedup / Number of Threads) × 100

# Data yang digunakan juga bervariasi 
data  besar : 19.326,
data sedang : 9.197,
data kecil : 4.507.

# Bubble sort

In [ ]:
import pandas as pd                 # digunakan untuk mengolah data tabel
import kagglehub                    # digunakan untuk mengambil dataset dari Kaggle
import time                         # digunakan untuk menghitung waktu eksekusi program
import threading                    # digunakan untuk membuat multi-thread
import tracemalloc                  # digunakan untuk mengukur penggunaan memori

from kagglehub import KaggleDatasetAdapter
from IPython.display import display

# =========================================================
# NAMA FILE DATASET
# =========================================================
file_path = "osn.csv"               # nama file dataset yang akan digunakan

# =========================================================
# LOAD DATASET
# =========================================================
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",     # link dataset di Kaggle 
    file_path,
)

# =========================================================
# MENGAMBIL KOLOM
# =========================================================
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]
# hanya mengambil kolom yang diperlukan agar data lebih sederhana

# =========================================================
# MENGISI DATA KOSONG
# =========================================================
df["Medali"] = df["Medali"].fillna("Partisipan")
# jika ada data medali yang kosong maka diisi dengan Partisipan

# =========================================================
# PRIORITAS MEDALI
# =========================================================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}
# semakin kecil nilainya maka semakin tinggi prioritasnya

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)
# mengubah nama medali menjadi angka prioritas

# Jika ada medali lain yang tidak dikenal
df["Prioritas"] = df["Prioritas"].fillna(99)
# jika ada data yang tidak masuk kategori maka diberi nilai 99

# =========================================================
# MENGUBAH DATAFRAME MENJADI LIST
# =========================================================
data = df.values.tolist()
# dataframe diubah menjadi list agar lebih mudah diproses bubble sort

# =========================================================
# FUNGSI BUBBLE SORT
# =========================================================
def bubble_sort(arr):

    n = len(arr)                    # menghitung jumlah data

    for i in range(n):              # perulangan sebanyak jumlah data

        for j in range(0, n - i - 1):
            # membandingkan elemen yang bersebelahan

            if arr[j][4] > arr[j + 1][4]:
                # index ke-4 adalah kolom prioritas

                arr[j], arr[j + 1] = arr[j + 1], arr[j]
                # menukar posisi jika data kiri lebih besar

    return arr                      # mengembalikan hasil sorting

# =========================================================
# SINGLE-THREAD EXECUTION
# =========================================================
single_data = data.copy()
# membuat salinan data agar data asli tidak berubah

tracemalloc.start()
# mulai mencatat penggunaan memori

start_single = time.time()
# mencatat waktu mulai

single_sorted = bubble_sort(single_data)
# proses sorting menggunakan 1 thread

end_single = time.time()
# mencatat waktu selesai

current, peak_single = tracemalloc.get_traced_memory()
# mengambil penggunaan memori maksimum

tracemalloc.stop()
# menghentikan pencatatan memori

single_time = end_single - start_single
# menghitung total waktu eksekusi

# =========================================================
# MULTI-THREAD EXECUTION
# =========================================================
multi_data = data.copy()
# membuat salinan data untuk pengujian multi-thread

mid = len(multi_data) // 2
# menentukan titik tengah data

part1 = multi_data[:mid]
# bagian pertama data

part2 = multi_data[mid:]
# bagian kedua data

# Thread Bubble Sort
thread1 = threading.Thread(target=bubble_sort, args=(part1,))
thread2 = threading.Thread(target=bubble_sort, args=(part2,))
# membuat dua thread yang menjalankan bubble sort

tracemalloc.start()
# mulai menghitung penggunaan memori

start_multi = time.time()
# waktu mulai multi-thread

thread1.start()
thread2.start()
# menjalankan kedua thread secara bersamaan

thread1.join()
thread2.join()
# menunggu sampai kedua thread selesai

merged = part1 + part2
# menggabungkan hasil sorting dari kedua thread

multi_sorted = bubble_sort(merged)
# sorting kembali agar seluruh data benar-benar urut

end_multi = time.time()
# waktu selesai multi-thread

current, peak_multi = tracemalloc.get_traced_memory()
# mengambil penggunaan memori maksimum

tracemalloc.stop()
# menghentikan pencatatan memori

multi_time = end_multi - start_multi
# menghitung waktu eksekusi multi-thread

# =========================================================
# PERFORMANCE IMPROVEMENT
# =========================================================
improvement = (
    (single_time - multi_time) / single_time
) * 100
# menghitung persentase peningkatan performa

# =========================================================
# MEMORY USAGE
# =========================================================
memory_usage_mb = peak_multi / (1024 * 1024)
# mengubah byte menjadi MB

# =========================================================
# DATAFRAME HASIL SORTING
# =========================================================
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)
# mengubah hasil sorting kembali menjadi dataframe

df_sorted = df_sorted.drop(columns=["Prioritas"])
# menghapus kolom prioritas karena hanya digunakan saat sorting

df_sorted.index = range(1, len(df_sorted) + 1)
# mengubah index agar dimulai dari angka 1

# =========================================================
# PENGATURAN TAMPILAN
# =========================================================
pd.set_option('display.max_rows', None)
# menampilkan seluruh baris

pd.set_option('display.max_columns', None)
# menampilkan seluruh kolom

pd.set_option('display.width', None)
# menyesuaikan lebar tampilan

# =========================================================
# MENAMPILKAN HASIL SORTING
# =========================================================
display(df_sorted)
# menampilkan data yang sudah diurutkan

# =========================================================
# TABEL PERFORMA
# =========================================================
performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Bubble Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 4)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 4)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 2)]
})
# membuat tabel hasil pengukuran performa

print("\nHASIL PENGUKURAN PERFORMA:\n")

display(performance_df)
# menampilkan tabel performa algoritma

# Merge sort

In [ ]:
import pandas as pd                     # untuk mengolah data dalam bentuk tabel
import kagglehub                        # untuk mengambil dataset dari Kaggle
import time                             # untuk menghitung waktu eksekusi
import threading                        # untuk menjalankan multi-thread
import tracemalloc                      # untuk mengukur penggunaan memori

from kagglehub import KaggleDatasetAdapter
from IPython.display import display     # untuk menampilkan dataframe dengan rapi

# =========================================================
# NAMA FILE DATASET
# =========================================================
file_path = "osn.csv"                   # nama file dataset yang digunakan

# =========================================================
# LOAD DATASET
# =========================================================
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",
    file_path,
)                                       # mengambil dataset OSN dari Kaggle

# =========================================================
# MENGAMBIL KOLOM
# =========================================================
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]
# hanya mengambil kolom yang diperlukan untuk proses sorting

# =========================================================
# MENGISI DATA KOSONG
# =========================================================
df["Medali"] = df["Medali"].fillna("Partisipan")
# jika ada data medali kosong maka diganti menjadi Partisipan

# =========================================================
# PRIORITAS MEDALI
# =========================================================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}
# membuat prioritas medali agar bisa dibandingkan dengan angka

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)
# mengubah nama medali menjadi nilai prioritas

# Jika ada medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)
# jika ada nilai yang tidak dikenali maka diberi nilai 99

# =========================================================
# MENGUBAH DATAFRAME MENJADI LIST
# =========================================================
data = df.values.tolist()
# dataframe diubah menjadi list agar mudah diproses merge sort

# =========================================================
# MERGE SORT
# =========================================================
def merge_sort(arr):

    if len(arr) <= 1:
        return arr
        # jika data hanya 1 atau kosong maka langsung dikembalikan

    mid = len(arr) // 2
    # mencari titik tengah data

    left = merge_sort(arr[:mid])
    # membagi bagian kiri lalu diurutkan secara rekursif

    right = merge_sort(arr[mid:])
    # membagi bagian kanan lalu diurutkan secara rekursif

    return merge(left, right)
    # menggabungkan dua bagian yang sudah terurut

# =========================================================
# MERGE FUNCTION
# =========================================================
def merge(left, right):

    hasil = []
    # list untuk menyimpan hasil penggabungan

    while len(left) > 0 and len(right) > 0:
    # selama kedua list masih memiliki data

        # Index 4 = Prioritas Medali
        if left[0][4] < right[0][4]:
        # membandingkan prioritas data paling depan

            hasil.append(left.pop(0))
            # mengambil data dari kiri jika lebih kecil

        else:

            hasil.append(right.pop(0))
            # mengambil data dari kanan jika lebih kecil

    # Sisa data
    hasil.extend(left)
    # menambahkan sisa data kiri

    hasil.extend(right)
    # menambahkan sisa data kanan

    return hasil
    # mengembalikan hasil penggabungan

# =========================================================
# SINGLE-THREAD EXECUTION
# =========================================================
single_data = data.copy()
# membuat salinan data agar data asli tidak berubah

tracemalloc.start()
# mulai mengukur penggunaan memori

start_single = time.time()
# mencatat waktu mulai

single_sorted = merge_sort(single_data)
# menjalankan merge sort dengan 1 thread

end_single = time.time()
# mencatat waktu selesai

current, peak_single = tracemalloc.get_traced_memory()
# mengambil penggunaan memori maksimum

tracemalloc.stop()
# menghentikan pengukuran memori

single_time = end_single - start_single
# menghitung total waktu eksekusi

# =========================================================
# MULTI-THREAD EXECUTION
# =========================================================
multi_data = data.copy()
# membuat salinan data untuk pengujian multi-thread

mid = len(multi_data) // 2
# mencari titik tengah data

part1 = multi_data[:mid]
# bagian pertama data

part2 = multi_data[mid:]
# bagian kedua data

hasil1 = []
hasil2 = []
# variabel untuk menyimpan hasil sorting tiap thread

# Fungsi thread
def sort_part1():
    global hasil1
    hasil1 = merge_sort(part1)
    # mengurutkan bagian pertama

def sort_part2():
    global hasil2
    hasil2 = merge_sort(part2)
    # mengurutkan bagian kedua

thread1 = threading.Thread(target=sort_part1)
# membuat thread pertama

thread2 = threading.Thread(target=sort_part2)
# membuat thread kedua

tracemalloc.start()
# mulai menghitung penggunaan memori

start_multi = time.time()
# mencatat waktu mulai multi-thread

# Menjalankan thread
thread1.start()
thread2.start()
# kedua thread dijalankan secara bersamaan

# Menunggu thread selesai
thread1.join()
thread2.join()
# program menunggu sampai kedua thread selesai

# Menggabungkan hasil
multi_sorted = merge(hasil1, hasil2)
# menggabungkan dua bagian yang sudah terurut

end_multi = time.time()
# mencatat waktu selesai

current, peak_multi = tracemalloc.get_traced_memory()
# mengambil penggunaan memori maksimum

tracemalloc.stop()
# menghentikan pengukuran memori

multi_time = end_multi - start_multi
# menghitung total waktu eksekusi multi-thread

# =========================================================
# PERFORMANCE IMPROVEMENT
# =========================================================
improvement = (
    (single_time - multi_time) / single_time
) * 100
# menghitung persentase peningkatan performa

# =========================================================
# MEMORY USAGE
# =========================================================
memory_usage_mb = peak_multi / (1024 * 1024)
# mengubah penggunaan memori dari byte ke MB

# =========================================================
# HASIL SORTING
# =========================================================
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)
# mengubah hasil sorting kembali menjadi dataframe

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])
# kolom prioritas tidak perlu ditampilkan

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)
# mengubah nomor index agar dimulai dari 1

# =========================================================
# PENGATURAN TAMPILAN
# =========================================================
pd.set_option('display.max_rows', None)
# menampilkan semua baris

pd.set_option('display.max_columns', None)
# menampilkan semua kolom

pd.set_option('display.width', None)
# menyesuaikan lebar tampilan

# =========================================================
# MENAMPILKAN HASIL SORTING
# =========================================================
display(df_sorted)
# menampilkan data yang sudah diurutkan

# =========================================================
# TABEL PERFORMA
# =========================================================
performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Merge Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 4)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 4)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 2)]
})
# membuat tabel hasil pengujian performa

print("\nHASIL PENGUKURAN PERFORMA:\n")

display(performance_df)
# menampilkan tabel performa merge sort

# Shell sort

# GAP pada shell sort 

 SHELL SORT
 Gap adalah jarak antar elemen yang dibandingkan.

 Alasan penggunaan gap:
 1. Gap besar mempercepat perpindahan elemen yang posisinya jauh dari tempat yang benar.
 2. Gap sedang mulai merapikan data secara bertahap.
 3. Gap kecil menyempurnakan hasil pengurutan.
 4. Gap terakhir harus bernilai 1 agar seluruh data terurut sempurna.
 5. Gap dapat disesuaikan dengan ukuran dataset:
    - Dataset kecil  : 5, 2, 1
    - Dataset sedang : 10, 5, 2, 1
    - Dataset besar  : n//2, n//4, ..., 1
 6. Semakin tepat pemilihan gap, semakin efisien proses sorting.

 Pada program ini digunakan gap n//2 yang kemudian dibagi 2
 hingga mencapai 1 karena sederhana dan umum digunakan.

kenapa kami mengubah gap pada shell sort, karena shell sort menurut kami Tidak memerlukan memori tambahan yang besar, 

dan juga sort ini lebih ceoat dibandingkan bubble

maka dari itu kami mengembangkan gap pada shell untuk mencari data sedang dan kecil pada data OSN 2024

In [ ]:
import pandas as pd                     # digunakan untuk mengolah data dalam bentuk tabel
import kagglehub                        # digunakan untuk mengambil dataset dari Kaggle
import time                             # digunakan untuk menghitung waktu eksekusi program
import threading                        # digunakan untuk menjalankan multi-thread
import tracemalloc                      # digunakan untuk mengukur penggunaan memori

from kagglehub import KaggleDatasetAdapter
from IPython.display import display     # digunakan untuk menampilkan dataframe

# =========================================================
# NAMA FILE DATASET
# =========================================================
file_path = "osn.csv"                   # nama file dataset yang digunakan

# =========================================================
# LOAD DATASET
# =========================================================
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "anakpindahan/indonesia-national-science-olympiad-osn/versions/10",
    file_path,
)
# mengambil dataset OSN dari Kaggle

# =========================================================
# MENGAMBIL KOLOM
# =========================================================
df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]
# hanya mengambil kolom yang diperlukan untuk sorting

# =========================================================
# MENGISI DATA KOSONG
# =========================================================
df["Medali"] = df["Medali"].fillna("Partisipan")
# jika ada nilai kosong pada kolom medali akan diganti menjadi Partisipan

# =========================================================
# PRIORITAS MEDALI
# =========================================================
prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}
# membuat urutan prioritas medali

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)
# mengubah nama medali menjadi angka prioritas

# Jika ada medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)
# jika ada data yang tidak dikenali maka diberi nilai 99

# =========================================================
# MENGUBAH DATAFRAME MENJADI LIST
# =========================================================
data = df.values.tolist()
# dataframe diubah menjadi list agar lebih mudah diproses shell sort

# =========================================================
# FUNGSI SHELL SORT
# =========================================================
def shell_sort(arr):

    n = len(arr)                       # menghitung jumlah data

    # =====================================================
    # GAP SEQUENCE
    # =====================================================
    #
    # Nilai gap menentukan jarak antar elemen yang dibandingkan.
    # Gap besar mempercepat perpindahan elemen jarak jauh,
    # sedangkan gap kecil menyempurnakan hasil pengurutan.
    #
    # Silakan ubah strategi gap sesuai ukuran dataset dan kebutuhan eksperimen.
    # Contoh:
    # gap = n // 2
    # gap = n // 3
    # gap = n // 4
    # atau gunakan custom gap sequence seperti [10, 5, 2, 1]
    #
    # Catatan:
    # Gap terakhir harus bernilai 1 agar data terurut sempurna.
    #
    gap = n // 2                       # menentukan gap awal

    while gap > 0:                     # perulangan selama gap masih lebih dari 0

        for i in range(gap, n):        # mulai dari indeks sesuai nilai gap

            temp = arr[i]              # menyimpan data sementara
            j = i

            # Index 4 = Prioritas Medali
            while j >= gap and arr[j - gap][4] > temp[4]:
                # membandingkan prioritas berdasarkan jarak gap

                arr[j] = arr[j - gap]
                # menggeser data ke kanan

                j -= gap
                # berpindah ke posisi sebelumnya sesuai gap

            arr[j] = temp
            # menempatkan data pada posisi yang benar

        gap //= 2
        # mengurangi nilai gap menjadi setengahnya

    return arr
    # mengembalikan hasil sorting

# =========================================================
# SINGLE-THREAD EXECUTION
# =========================================================
single_data = data.copy()
# membuat salinan data untuk pengujian single-thread

tracemalloc.start()
# mulai menghitung penggunaan memori

start_single = time.time()
# mencatat waktu mulai

single_sorted = shell_sort(single_data)
# menjalankan shell sort menggunakan satu thread

end_single = time.time()
# mencatat waktu selesai

current, peak_single = tracemalloc.get_traced_memory()
# mengambil penggunaan memori maksimum

tracemalloc.stop()
# menghentikan pencatatan memori

single_time = end_single - start_single
# menghitung total waktu eksekusi

# =========================================================
# MULTI-THREAD EXECUTION
# =========================================================
multi_data = data.copy()
# membuat salinan data untuk pengujian multi-thread

# Membagi data menjadi 2 bagian
mid = len(multi_data) // 2
# mencari titik tengah data

part1 = multi_data[:mid]
# bagian pertama data

part2 = multi_data[mid:]
# bagian kedua data

hasil1 = []
hasil2 = []
# menampung hasil sorting masing-masing thread

# Fungsi thread
def sort_part1():
    global hasil1
    hasil1 = shell_sort(part1)
    # mengurutkan bagian pertama data

def sort_part2():
    global hasil2
    hasil2 = shell_sort(part2)
    # mengurutkan bagian kedua data

thread1 = threading.Thread(target=sort_part1)
# membuat thread pertama

thread2 = threading.Thread(target=sort_part2)
# membuat thread kedua

tracemalloc.start()
# mulai menghitung penggunaan memori

start_multi = time.time()
# mencatat waktu mulai multi-thread

# Menjalankan thread
thread1.start()
thread2.start()
# kedua thread dijalankan secara bersamaan

# Menunggu thread selesai
thread1.join()
thread2.join()
# program menunggu sampai kedua thread selesai

# Menggabungkan hasil
merged = hasil1 + hasil2
# menggabungkan hasil sorting dari kedua thread

# Sorting akhir
multi_sorted = shell_sort(merged)
# sorting ulang agar seluruh data benar-benar terurut

end_multi = time.time()
# mencatat waktu selesai

current, peak_multi = tracemalloc.get_traced_memory()
# mengambil penggunaan memori maksimum

tracemalloc.stop()
# menghentikan pencatatan memori

multi_time = end_multi - start_multi
# menghitung total waktu eksekusi multi-thread

# =========================================================
# PERFORMANCE IMPROVEMENT
# =========================================================
improvement = (
    (single_time - multi_time) / single_time
) * 100
# menghitung persentase peningkatan performa

# =========================================================
# MEMORY USAGE
# =========================================================
memory_usage_mb = peak_multi / (1024 * 1024)
# mengubah satuan byte menjadi MB

# =========================================================
# HASIL SORTING
# =========================================================
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)
# mengubah hasil sorting menjadi dataframe

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])
# kolom prioritas tidak perlu ditampilkan

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)
# mengubah index agar dimulai dari angka 1

# =========================================================
# PENGATURAN TAMPILAN
# =========================================================
pd.set_option('display.max_rows', None)
# menampilkan semua baris

pd.set_option('display.max_columns', None)
# menampilkan semua kolom

pd.set_option('display.width', None)
# menyesuaikan lebar tampilan

# =========================================================
# MENAMPILKAN HASIL SORTING
# =========================================================
display(df_sorted)
# menampilkan data yang sudah diurutkan

# =========================================================
# TABEL PERFORMA
# =========================================================
performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Shell Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 4)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 4)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 2)]
})
# membuat tabel hasil pengujian performa

print("\nHASIL PENGUKURAN PERFORMA:\n")

display(performance_df)
# menampilkan tabel hasil pengukuran performa

# Shell sort dengan data kecil dan sedang 

In [ ]:
import pandas as pd
import time
import threading
import tracemalloc
from IPython.display import display

# NAMA FILE CSV
# file_path = "dataKecil.csv"           # menggunakan dataset kecil untuk pengujian awal
file_path = "dataSeddang.csv"           # menggunakan dataset besar untuk pengujian akhir


df = pd.read_csv("dataKecil.csv", sep=";") 

df = df[["Medali", "Nama Peserta", "Bidang", "Provinsi"]]

df["Medali"] = df["Medali"].fillna("Partisipan") # mengisi data kosong dengan "Partisipan"

prioritas_medali = {
    "Emas": 1,
    "Perak": 2,
    "Perunggu": 3,
    "Harapan": 4,
    "Partisipan": 5
}

# Menambahkan kolom prioritas
df["Prioritas"] = df["Medali"].map(prioritas_medali)

# Jika ada nilai medali lain
df["Prioritas"] = df["Prioritas"].fillna(99)
data = df.values.tolist()

def shell_sort(arr):

    n = len(arr)
    gap = n // 2

    while gap > 0:

        for i in range(gap, n):

            temp = arr[i]
            j = i

            while j >= gap and arr[j - gap][4] > temp[4]:

                arr[j] = arr[j - gap]
                j -= gap

            arr[j] = temp

        gap //= 2

    return arr

# SINGLE-THREAD EXECUTION
single_data = data.copy()

tracemalloc.start()

start_single = time.time()

single_sorted = shell_sort(single_data)

end_single = time.time()

current, peak_single = tracemalloc.get_traced_memory()

tracemalloc.stop()

single_time = end_single - start_single

# MULTI-THREAD EXECUTION
multi_data = data.copy()

mid = len(multi_data) // 2

part1 = multi_data[:mid]
part2 = multi_data[mid:]

hasil1 = []
hasil2 = []

# FUNGSI THREAD
def sort_part1():
    global hasil1
    hasil1 = shell_sort(part1)

def sort_part2():
    global hasil2
    hasil2 = shell_sort(part2)

thread1 = threading.Thread(target=sort_part1)
thread2 = threading.Thread(target=sort_part2)

tracemalloc.start()

start_multi = time.time()

thread1.start()
thread2.start()

thread1.join()
thread2.join()

merged = hasil1 + hasil2 # MENGGABUNGKAN HASIL

# Sorting akhir
multi_sorted = shell_sort(merged)

end_multi = time.time()

current, peak_multi = tracemalloc.get_traced_memory()

tracemalloc.stop()

multi_time = end_multi - start_multi

improvement = (
    (single_time - multi_time) / single_time
) * 100


memory_usage_mb = peak_multi / (1024 * 1024) # rumus untuk mengubah byte ke megabyte
                                                # Memory Usage (MB) = Peak Memory (Byte) 1024 * 1024

# HASIL SORTING
df_sorted = pd.DataFrame(
    multi_sorted,
    columns=[
        "Medali",
        "Nama Peserta",
        "Bidang",
        "Provinsi",
        "Prioritas"
    ]
)

# Menghapus kolom prioritas
df_sorted = df_sorted.drop(columns=["Prioritas"])

# Index mulai dari 1
df_sorted.index = range(1, len(df_sorted) + 1)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

print("\nHASIL SORTING SHELL SORT:\n")
display(df_sorted)

performance_df = pd.DataFrame({
    "Sorting Algorithm": ["Shell Sort"],
    "Single-thread Execution Time (seconds)": [round(single_time, 6)],
    "Multi-thread Execution Time (seconds)": [round(multi_time, 6)],
    "Performance Improvement (%)": [round(improvement, 2)],
    "Memory Usage (MB)": [round(memory_usage_mb, 4)]
})

print("\nHASIL PENGUKURAN PERFORMA:\n")
display(performance_df)